# Reproduce the solar filament submission

Run these cells in order from the repository root. The models read the official test JPEGs and draw a separate mask for each predicted filament. The submitted file contains 1,262 instances across 180 observations. The Kaggle public leaderboard displayed a panoptic quality score of 0.37.

## Find the test observations

Set `SOLAR_TEST_IMAGES` to the test-image directory or edit `image_dir` in the next cell. The default paths cover an unpacked competition archive and Kaggle's competition input mount. Set `SOLAR_OUTPUT_CSV` to change the output destination.

In [ ]:
import os
import sys
from pathlib import Path

repo = Path(os.environ.get("SOLAR_REPOSITORY", Path.cwd())).resolve()
if not (repo / "solar_filaments.py").is_file():
    raise FileNotFoundError("Open the notebook from the repository root or set SOLAR_REPOSITORY.")
sys.path.insert(0, str(repo))

dataset_folder = Path("MAGFiLO_1.0_Kaggle_2026") / "test" / "test_images"
locations = [
    repo / "data" / dataset_folder,
    repo / dataset_folder,
    repo / "test_images",
    Path("/kaggle/input/competitions/filament-segmentation-2026") / dataset_folder,
    Path("/kaggle/input/filament-segmentation-2026") / dataset_folder,
]
image_dir = Path(os.environ["SOLAR_TEST_IMAGES"]) if "SOLAR_TEST_IMAGES" in os.environ else next(
    (candidate for candidate in locations if candidate.is_dir()), locations[0]
)
if not image_dir.is_dir():
    raise FileNotFoundError(f"Set SOLAR_TEST_IMAGES to the official test_images directory. Tried {image_dir}.")
images = sorted(path for path in image_dir.iterdir() if path.suffix.lower() in {".jpg", ".jpeg"})
if len(images) != 180 or len({path.stem for path in images}) != 180:
    raise ValueError(f"Expected 180 distinct official JPEGs, found {len(images)}.")
output_csv = Path(os.environ.get("SOLAR_OUTPUT_CSV", "output/submission.csv")).resolve()
print(f"{len(images)} observations. Output: {output_csv}")

## Load the released weights

The YOLO11m-seg checkpoint proposes boxes on each full frame at 1,536 pixels. The refiner uses a ResNet-18 encoder and three inputs for each 256 × 256 crop: grayscale intensity, local histogram equalization (CLAHE), and a rectangular box seed. It averages probabilities over four flip views, maps them back to the 2,048 × 2,048 frame and thresholds them at 0.55.

In [ ]:
import torch
from solar_filaments import FilamentPredictor, run_submission

if not torch.cuda.is_available():
    raise RuntimeError("Reproduction requires a CUDA-enabled PyTorch build and NVIDIA GPU.")
device = "cuda"
weights_dir = repo / "weights"
predictor = FilamentPredictor(weights_dir=weights_dir, device=device)
print(torch.cuda.get_device_name(0), torch.__version__, torch.version.cuda)
print("Released weights:", ", ".join(path.name for path in sorted(weights_dir.glob("*.pt"))))

## Trace one observation

Detections scoring at least 0.30 receive first claim on pixels. The second pass considers boxes scoring from 0.10 to 0.30. A ridge model filters those extra masks at 0.24. Four small spatial heads then score the accepted masks. Their mean expected match contribution must reach 0.14547 to retain an instance. The table printed below traces these decisions for the first test image; the cyan outlines show its retained masks.

In [ ]:
import cv2
import numpy as np
from PIL import Image
from IPython.display import display

example = images[0]
labels, counts = predictor.predict_image(example)
assert labels.shape == (2048, 2048)
print(example.stem)
for stage, count in counts.items():
    print(f"{stage:23s} {count}")

with Image.open(example) as source:
    overlay = np.asarray(source.convert("RGB")).copy()
for label in range(1, int(labels.max()) + 1):
    contours, _ = cv2.findContours((labels == label).astype(np.uint8), cv2.RETR_EXTERNAL,
                                  cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(overlay, contours, -1, (39, 231, 241), 3)
display(Image.fromarray(overlay).resize((640, 640), Image.Resampling.LANCZOS))

## Run all 180 observations

The exporter assigns consecutive instance numbers within each image and encodes every mask as a COCO compressed RLE at native resolution. It checks that each RLE decodes to its original mask, that instance IDs are unique and that no two masks claim the same pixel. An observation with zero predictions stays in the processing manifest and contributes zero CSV rows.

In [ ]:
output_csv.parent.mkdir(parents=True, exist_ok=True)
receipt = run_submission(image_dir, output_csv, weights_dir=weights_dir, device=device,
                         expected_images=180, predictor=predictor)
print({key: receipt[key] for key in ("processed_images", "rows", "unique_ids",
                                     "images_represented_in_csv", "zero_prediction_images")})

## Compare with the submitted file

The digest refers to the complete CSV, including its row order, header and line endings. The reference hardware and software versions are listed in the README.

In [ ]:
import csv
import hashlib

expected_sha256 = "a0324008edfd7a4ea14a990a03e27667269da93475d5c096729003f90844f93e"
with output_csv.open(newline="", encoding="utf-8") as stream:
    reader = csv.DictReader(stream)
    assert reader.fieldnames == ["filament_id", "segmentation_rle"]
    instance_ids = [row["filament_id"] for row in reader]
assert receipt["processed_images"] == 180
assert len(instance_ids) == len(set(instance_ids)) == receipt["rows"] == 1262
assert len(receipt["zero_prediction_images"]) == 2
assert receipt["all_rles_roundtrip"] and receipt["disjoint_instances"]
digest = hashlib.sha256(output_csv.read_bytes()).hexdigest()
assert digest == receipt["csv_sha256"]
assert digest == expected_sha256, f"CSV differs from the reference: {digest}"
print(f"Verified {len(instance_ids)} unique instances in {output_csv.name}. SHA-256 {digest}")